In [1]:
import pandas as pd

In [2]:
pd.options.display.max_columns=None
pd.options.display.precision=4

In [3]:
df = pd.read_csv("data/train.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594194 entries, 0 to 594193
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                594194 non-null  int64  
 1   gender            594194 non-null  object 
 2   SeniorCitizen     594194 non-null  int64  
 3   Partner           594194 non-null  object 
 4   Dependents        594194 non-null  object 
 5   tenure            594194 non-null  int64  
 6   PhoneService      594194 non-null  object 
 7   MultipleLines     594194 non-null  object 
 8   InternetService   594194 non-null  object 
 9   OnlineSecurity    594194 non-null  object 
 10  OnlineBackup      594194 non-null  object 
 11  DeviceProtection  594194 non-null  object 
 12  TechSupport       594194 non-null  object 
 13  StreamingTV       594194 non-null  object 
 14  StreamingMovies   594194 non-null  object 
 15  Contract          594194 non-null  object 
 16  PaperlessBilling  59

In [5]:
df.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,No,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,Yes,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [ ]:
print("Class distribution:")
print(df['Churn'].value_counts())
print(f"Churn rate: {df['Churn'].mean()}")

# scale_pos_weight = (y == 0).sum() / (y == 1).sum()
# print(f"\nscale_pos_weight: {scale_pos_weight:.4f}")

Class distribution:
Churn
No     460377
Yes    133817
Name: count, dtype: int64


In [6]:
for col in df.select_dtypes(include=['object']).columns:
    print(f"Column: {col}")
    unique_vals = df[col].unique()
    print(f"Categories: {unique_vals}")
    print(f"Number of categories: {len(unique_vals)}")
    print("-" * 50)

Column: gender
Categories: ['Male' 'Female']
Number of categories: 2
--------------------------------------------------
Column: Partner
Categories: ['Yes' 'No']
Number of categories: 2
--------------------------------------------------
Column: Dependents
Categories: ['Yes' 'No']
Number of categories: 2
--------------------------------------------------
Column: PhoneService
Categories: ['Yes' 'No']
Number of categories: 2
--------------------------------------------------
Column: MultipleLines
Categories: ['No' 'Yes' 'No phone service']
Number of categories: 3
--------------------------------------------------
Column: InternetService
Categories: ['DSL' 'Fiber optic' 'No']
Number of categories: 3
--------------------------------------------------
Column: OnlineSecurity
Categories: ['Yes' 'No' 'No internet service']
Number of categories: 3
--------------------------------------------------
Column: OnlineBackup
Categories: ['No' 'Yes' 'No internet service']
Number of categories: 3
--------

## Split Data

Production-Grade Rule

Follow this order:

1. Split data (train / validation / test)
2. Fit feature engineering using TRAIN only
3. Apply transformations to val/test
4. Train model

Why? **To prevent leakage.**

In [ ]:
from sklearn.model_selection import train_test_split

import numpy as np

In [ ]:
train_size = 0.7
val_size = 0.15
test_size = 0.15

# Create train/val/test splits: 70% traun, 15% val, and 15% test
X = df.drop(columns=['id', 'Churn'])
y = (df['Churn'] == 'Yes').astype(int)

# First split: 85% train+val, 15% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)

# Second split: split temp into train/val
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=(0.15 / 0.85), stratify=y_temp, random_state=42
)

print(f"Train shape: {X_train.shape}")
print(f"Val shape: {X_val.shape}")
print(f"Test shape: {X_test.shape}")

print('Split ratios:')
# Train = 64%
# Val   = 16%
# Test  = 20%

print("\nClass balance:")
print(y_train.value_counts(normalize=True))
print(y_val.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

Train shape: (380284, 19)
Val shape: (95071, 19)
Test shape: (118839, 19)
Split ratios:

Class balance:
Churn
0    0.7748
1    0.2252
Name: proportion, dtype: float64
Churn
0    0.7748
1    0.2252
Name: proportion, dtype: float64
Churn
0    0.7748
1    0.2252
Name: proportion, dtype: float64


Hidden issue: category consistency across splits

Right now:

Train categories ≠ Test categories (possible)

Example:

Train: PaymentMethod = {A, B, C}
Test: PaymentMethod = {A, B, C, D}
Good news:

✔️ CatBoost handles unseen categories

# Evaluation
evaluate on area under the ROC curve between the predicted probability and the observed target.

## Feature Engineering Pipeline

In [ ]:
df["Contract"].unique()



In [ ]:
def create_features(df):

    df = df.copy()

    # 1. Tenure bins
    df["tenure_group"] = pd.cut(
        df["tenure"],
        bins=[0, 6, 12, 24, 48, 72],
        labels=["0-6", "6-12", "12-24", "24-48", "48-72"],
    )

    # 2. Service count
    service_cols = [
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
    ]

    df["num_services"] = df[service_cols].eq("Yes").sum(axis=1)

    # 3. Long contract
    df["is_long_contract"] = df["Contract"].isin(["One year", "Two year"]).astype(int)

    # 4. Has phone
    df["has_phone"] = (df["PhoneService"] == "Yes").astype(int)

    # 5. Total services
    df["total_services"] = df["num_services"] + df["has_phone"]

    # 6. Charges per service
    df["charges_per_service"] = df["MonthlyCharges"] / (df["num_services"] + 1)

    # 7. Avg monthly spend
    df["avg_monthly_spend"] = df["TotalCharges"] / (df["tenure"] + 1)

    # 8. Paperless + month-to-month
    df["paperless_monthly"] = (
        (df["PaperlessBilling"] == "Yes") & (df["Contract"] == "Month-to-month")
    ).astype(int)

    df["new_customer_monthly"] = (
        (df["tenure"] < 12) & (df["Contract"] == "Month-to-month")
    ).astype(int)

    return df


In [ ]:
X_train = create_features(X_train)
X_val = create_features(X_val)
X_test = create_features(X_test)


# Features That Need Train-Only Statistics
median_charge = X_train["MonthlyCharges"].median()

X_train["high_value_customer"] = (X_train["MonthlyCharges"] > median_charge).astype(int)

X_val["high_value_customer"] = (X_val["MonthlyCharges"] > median_charge).astype(int)

X_test["high_value_customer"] = (X_test["MonthlyCharges"] > median_charge).astype(int)


cat_features = X_train.columns[X_train.dtypes == "object"].tolist()

# For CatBoost
for col in cat_features:
    X_train[col] = X_train[col].astype(str)
    X_val[col] = X_val[col].astype(str)
    X_test[col] = X_test[col].astype(str)


In [ ]:
# # Check assumption: When PhoneService == 'No', related columns have 'No phone service'
# # And vice versa

# # Rule: If PhoneService == 'No' → MultipleLines must be 'No phone service'
# invalid_multiline_given_no_phone = df.loc[
#     (df['PhoneService'] == 'No') &
#     (df['MultipleLines'] != 'No phone service')
# ]

# print(f"Violations (No PhoneService → invalid MultipleLines): {len(invalid_multiline_given_no_phone)}")


# # Reverse Rule: If MultipleLines == 'No phone service' → PhoneService must be 'No'
# invalid_phone_given_no_multiline = df.loc[
#     (df['MultipleLines'] == 'No phone service') &
#     (df['PhoneService'] != 'No')
# ]

# print(f"Violations (No phone service label → invalid PhoneService): {len(invalid_phone_given_no_multiline)}")

Violations (No PhoneService → invalid MultipleLines): 0
Violations (No phone service label → invalid PhoneService): 0


In [9]:
df.columns

Index(['id', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

For low-cardinality categorical features:
→ ALL models (XGB, LGBM, CatBoost) can perform well

The difference comes from:
→ how safely and correctly you handle categories

Start with CatBoost

# Train Model

CatBoost does not process categorical features in any specific way.

CatBoost interprets the value of a numerical feature as a missing value if it is equal to one of the following values, which are package-dependant:



## 1. CatBoost

In [6]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, recall_score, precision_score, accuracy_score


In [ ]:

model = CatBoostClassifier(
    iterations=100, learning_rate=0.1, depth=6, eval_metric="AUC", verbose=0
)
model.fit(
    X_train, y_train, cat_features=cat_features
)

y_pred = model.predict(X_val)

train_prob_1 = model.predict_proba(X_train)[:, 1]
y_prob_1 = model.predict_proba(X_val)[:, 1]

train_roc = roc_auc_score(y_train, train_prob_1)
val_roc = roc_auc_score(y_val, y_prob_1)
precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)
accuracy = accuracy_score(y_val, y_pred)

print("Train ROC:", train_roc)
print("Validation ROC:", val_roc)
print("Precision:", precision)
print("Recall:", recall)
print("Accuracy:", accuracy)

Train ROC: 0.9138518891577784
Validation ROC: 0.9131914466812736
Precision: 0.7045812782834189
Recall: 0.6471906963710242
Accuracy: 0.8594313723427754


In [16]:
model.get_feature_importance(prettified=True)


,Feature Id,Importances
0,tenure,25.8252
1,Contract,22.5635
2,MonthlyCharges,22.3556
3,PaymentMethod,8.9571
4,TotalCharges,4.6515
5,OnlineSecurity,3.1441
6,PaperlessBilling,2.5971
7,SeniorCitizen,2.1170
8,Dependents,1.5038
9,TechSupport,1.4502


1. When model predicts churn: 70% actually churn
2. Model catches 65% of churners.
    35% churners missed

### Model Refinement Stage

In [ ]:
# Step 1 — Define Optuna Objective (Inner CV)

import optuna
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

def objective(trial):

    print(f"Trial {trial.number} started")

    params = {
        "iterations": trial.suggest_int("iterations", 100, 600),  # reduced
        "depth": trial.suggest_int("depth", 4, 8),  # reduced
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
        "random_strength": trial.suggest_float("random_strength", 0, 2),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 1),
        "border_count": trial.suggest_int("border_count", 32, 128),  # reduced
        "eval_metric": "AUC",
        "verbose": 0,
        "early_stopping_rounds": 50,  # added
    }

    # reduced folds (5 → 3)
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    scores = []

    for train_idx, val_idx in inner_cv.split(X_train, y_train):
        X_tr, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = CatBoostClassifier(**params)

        model.fit(
            X_tr,
            y_tr,
            eval_set=(X_val_fold, y_val_fold),  # required for early stopping
            cat_features=cat_features,
            verbose=0,
        )

        prob_1 = model.predict_proba(X_val_fold)[:, 1]
        score = roc_auc_score(y_val_fold, prob_1)

        scores.append(score)

        print(f"Trial {trial.number} finished with score: {score:.4f}")

    return np.mean(scores)

In [ ]:
# Step 2 — Nested CV (Outer Loop)

outer_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

outer_scores = []

for train_idx, test_idx in outer_cv.split(X_train, y_train):
    X_outer_train = X_train.iloc[train_idx]
    y_outer_train = y_train.iloc[train_idx]

    X_outer_val = X_train.iloc[test_idx]
    y_outer_val = y_train.iloc[test_idx]

    # Faster Optuna settings
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),  # smarter sampling
    )
    
    optuna.logging.set_verbosity(optuna.logging.INFO)
    study.optimize(objective, n_trials=15)  # slightly increased but still fast

    best_params = study.best_params

    model = CatBoostClassifier(**best_params, eval_metric="AUC", verbose=0)

    model.fit(X_outer_train, y_outer_train, cat_features=cat_features)

    prob_1 = model.predict_proba(X_outer_val)[:, 1]
    score = roc_auc_score(y_outer_val, prob_1)

    outer_scores.append(score)

print("Nested CV ROC AUC:")
print("Mean:", np.mean(outer_scores))
print("Std:", np.std(outer_scores))

[I 2026-03-25 09:56:45,206] A new study created in memory with name: no-name-13aa3592-74c2-4990-b835-9b2edcb27b4a


Trial 0 started
Trial 0 finished with score: 0.9145
Trial 0 finished with score: 0.9138


[I 2026-03-25 10:08:22,735] Trial 0 finished with value: 0.9143300021169284 and parameters: {'iterations': 287, 'depth': 8, 'learning_rate': 0.08960785365368121, 'l2_leaf_reg': 6.387926357773329, 'random_strength': 0.31203728088487304, 'bagging_temperature': 0.15599452033620265, 'border_count': 37}. Best is trial 0 with value: 0.9143300021169284.


Trial 0 finished with score: 0.9147
Trial 1 started
Trial 1 finished with score: 0.9147
Trial 1 finished with score: 0.9144


[I 2026-03-25 10:27:25,478] Trial 1 finished with value: 0.9147990433375427 and parameters: {'iterations': 533, 'depth': 7, 'learning_rate': 0.08341106432362087, 'l2_leaf_reg': 1.185260448662222, 'random_strength': 1.9398197043239886, 'bagging_temperature': 0.8324426408004217, 'border_count': 52}. Best is trial 1 with value: 0.9147990433375427.


...

Trial 11 finished with score: 0.9149
Trial 12 started
Trial 12 finished with score: 0.9147
Trial 12 finished with score: 0.9142
Trial 12 finished with score: 0.9154


[I 2026-03-25 14:13:18,589] Trial 12 finished with value: 0.9147692378364384 and parameters: {'iterations': 436, 'depth': 7, 'learning_rate': 0.1553212641487403, 'l2_leaf_reg': 9.98239653032408, 'random_strength': 1.253941600003122, 'bagging_temperature': 0.7604259435508065, 'border_count': 57}. Best is trial 1 with value: 0.9147990433375427.


Trial 13 started
Trial 13 finished with score: 0.9147
Trial 13 finished with score: 0.9143


[I 2026-03-25 14:26:48,524] Trial 13 finished with value: 0.9147445830098638 and parameters: {'iterations': 495, 'depth': 7, 'learning_rate': 0.17009575987734887, 'l2_leaf_reg': 8.212143895742859, 'random_strength': 1.9600623457880522, 'bagging_temperature': 0.7719295837601934, 'border_count': 54}. Best is trial 1 with value: 0.9147990433375427.


Trial 13 finished with score: 0.9153
Trial 14 started
Trial 14 finished with score: 0.9150


Use Cases: Highly effective when labeling is expensive or time-consuming (e.g., medical imaging, text analysis, fraud detection).

Techniques:
Pseudo-labeling: A model trains on labeled data, predicts labels for unlabeled data (pseudo-labels), and then retrains on the combined dataset.

Self-Training: A model uses its own predictions to teach itself iteratively.

Graph-based models: Leverages data relationships to propagate labels.
Advantages & Challenges: While it increases efficiency and reduces costs, semi-supervised learning relies on strong assumptions about data distribution, such as assuming data points in the same cluster share the same label. 
YouTube
YouTube
 +6


In [ ]:
# Step 3 — Train Final Model Using Best Params

# best_params = study.best_params

best_params = {'iterations': 474, 'depth': 6, 'learning_rate': 0.13296599086216027, 'l2_leaf_reg': 3.519918777432741, 'random_strength': 1.2383357548081855, 'bagging_temperature': 0.7833743713800178, 'border_count': 59}

final_model = CatBoostClassifier(**best_params, eval_metric="AUC", verbose=100)

final_model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_test, y_test),
    early_stopping_rounds=50,
    verbose=100
)

# Then evaluate
test_pred = final_model.predict(X_test)
test_prob_1 = final_model.predict_proba(X_test)[:, 1]
train_prob_1 = final_model.predict_proba(X_train)[:, 1]

train_roc = roc_auc_score(y_train, train_prob_1)
test_roc = roc_auc_score(y_test, test_prob_1)

precision = precision_score(y_test, test_pred)
recall = recall_score(y_test, test_pred)
accuracy = accuracy_score(y_test, test_pred)

print("Final model performance:\n")
print("Test ROC:", test_roc)
print("Train ROC:", train_roc)
# print("Precision:", precision)
# print("Recall:", recall)
# print("Accuracy:", accuracy)



0:	total: 791ms	remaining: 6m 14s
100:	total: 1m 1s	remaining: 3m 47s
200:	total: 2m 19s	remaining: 3m 9s
300:	total: 3m 20s	remaining: 1m 55s
400:	total: 4m 42s	remaining: 51.4s
473:	total: 5m 39s	remaining: 0us
Final model performance:

Test ROC: 0.9156430730847838
Train ROC: 0.9180600636847147
Precision: 0.7103822870884816
Recall: 0.6401748682883085
Accuracy: 0.86018899519518


## 2. LightGBM

In [ ]:
for col in cat_features:
    X_train[col] = X_train[col].astype("category")
    X_test[col] = X_test[col].astype("category")

# optuna + cv

# final training
best_params = study.best_params

lgb_model = lgb.LGBMClassifier(
    **best_params, objective="binary", metric="auc", random_state=42
)

lgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(50)],
)

# eval
lgb_train_prob = lgb_model.predict_proba(X_train)[:, 1]
lgb_test_prob = lgb_model.predict_proba(X_test)[:, 1]

train_roc = roc_auc_score(y_train, lgb_train_prob)
test_roc = roc_auc_score(y_test, lgb_test_prob)

print("LightGBM Performance\n")
print("Train ROC:", train_roc)
print("Test ROC:", test_roc)

In [ ]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_iris


def objective(trial):
    iris = load_iris()
    X, y = iris.data, iris.target

    # Suggest hyperparameters
    n_estimators = trial.suggest_int("n_estimators", 10, 100)
    max_depth = trial.suggest_int("max_depth", 2, 32)

    # Initialize model with suggested hyperparameters
    clf = RandomForestClassifier(
        n_estimators=n_estimators, max_depth=max_depth, random_state=42
    )

    # Perform cross-validation and return the mean score
    score = cross_val_score(clf, X, y, cv=3, n_jobs=-1).mean()
    return score


# Create a study object and optimize the objective function
study = optuna.create_study(direction="maximize")  # We want to maximize accuracy
study.optimize(objective, n_trials=100)

print("Best hyperparameters:", study.best_params)
print("Best CV score:", study.best_value)


In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV

# Define model
rf = RandomForestClassifier()

# Hyperparameter grid
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5, 10],
}

# CV loops
outer_cv = KFold(n_splits=3, shuffle=True, random_state=0)
inner_cv = KFold(n_splits=5, shuffle=True, random_state=0)

# Inner CV for hyperparameter tuning
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=inner_cv, n_jobs=-1)

# Outer CV for performance evaluation
scores = cross_val_score(grid_search, X, y, scoring="accuracy", cv=outer_cv, n_jobs=-1)

print("Accuracy: %.3f (%.3f)" % (np.mean(scores), np.std(scores)))


## 1️⃣ Categorical encoding
where it fails:
❌ Leakage via target encoding
❌ Train/serve mismatch
❌ High-cardinality + one-hot

Solution:
- Fit encoders ONLY on training folds (CV-safe)
- Define behavior for unseen categories (e.g. “other”)
- Prefer CatBoost or regularized target encoding

## 2️⃣ High cardinality
Where it fails:
❌ Label encoding trap
user_id: A → 1, B → 2, C → 3

Tree thinks:

user_id < 2.5

👉 Fake ordering → meaningless splits

❌ Target encoding trap (again)
❌ Memory still explodes

SOlution:
- Drop useless high-card features (IDs)
- Group categories (top-K + "other")
- Use CatBoost (native handling)
- Regularized target encoding

## 3️⃣ Feature leakage
Where it fails silently

Leakage is often non-obvious:

❌ Time leakage
❌ Aggregation leakage
❌ Pipeline leakage (scaling, encoding done before split)

Solution:
- Always split FIRST, then transform
- Use pipelines
- Use time-based validation when needed
- Think: "Would this feature exist at prediction time?"

> 👉 “Being careful” = having a strict pipeline discipline

## 4️⃣ Overfitting
Where this fails

Optuna finds best params for your validation setup

If validation is flawed:
Optuna → optimizes leakage or noise

❌ Example
Random CV on time data
Optuna tunes model
Looks great
Fails in production

Solution
- Correct validation strategy FIRST
- Then tune (Optuna, etc.)
- Use early stopping

> Bad validation → perfectly tuned bad model

## 5️⃣ Features MUST have signal
Signal = data problem, not model problem

Fix by:

better features
better data collection
domain understanding

## 6️⃣ Correlated features

In trees:
Correlation ≠ big problem

BUT:
❌ Feature importance becomes misleading
income and salary (highly correlated)

Tree may:
- pick one
- ignore the other
👉 Importance gets split or biased

❌ Model instability

Small data change:
- tree picks different correlated feature
- structure changes

Solution:
- Not critical to remove
- But be careful interpreting importance
- Use permutation importance

## Before training, always ask:

1. Was this feature available at prediction time?
2. Did I compute this using only past data?
3. Did I split BEFORE transformations?
4. Is my validation realistic?

Be Thinking:
> "I will design a pipeline where issues cannot happen easily"

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV

# Define model
rf = RandomForestClassifier()

# Hyperparameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5, 10]
}

# CV loops
outer_cv = KFold(n_splits=3, shuffle=True, random_state=0)
inner_cv = KFold(n_splits=5, shuffle=True, random_state=0)

# Inner CV for hyperparameter tuning
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=inner_cv, n_jobs=-1)

# Outer CV for performance evaluation
scores = cross_val_score(grid_search, X, y, scoring='accuracy', cv=outer_cv, n_jobs=-1)

print('Accuracy: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))